In [1]:
import pandas as pd

df = pd.read_parquet("../data/yellow_tripdata_2023-01.parquet")

print(df.columns)

df.head()

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee'],
      dtype='object')


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


## Goal of this project is to predict demand for particular location.

1. We only care about pick up dime and time and location. 
2. Then we aggregate it to get count of how many pick ups we get from at each location at paticular time. 
3. Create hour, day of the week, month, weekend, pick up count in a hour past, pick up count in 24 hour past, and rolling average of 24 hour past. 
4. split 80% for training and split 
4. Use these column to predict pick up count using random forest. 

In [2]:
df = df[["tpep_pickup_datetime", "PULocationID"]]

In [3]:
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.floor("h")

In [4]:
demand = (
    df.groupby(["pickup_hour", "PULocationID"])
      .size()
      .reset_index(name="pickup_count")
)

In [5]:
demand.head()

,pickup_hour,PULocationID,pickup_count
0,2008-12-31 23:00:00,7,1
1,2008-12-31 23:00:00,132,1
2,2022-10-24 17:00:00,1,1
3,2022-10-24 20:00:00,17,1
4,2022-10-24 21:00:00,48,1


In [6]:
demand = demand.sort_values(["PULocationID", "pickup_hour"])

In [7]:
demand.head()

,pickup_hour,PULocationID,pickup_count
2,2022-10-24 17:00:00,1,1
8,2022-10-25 03:00:00,1,1
623,2023-01-01 05:00:00,1,1
916,2023-01-01 08:00:00,1,1
1224,2023-01-01 11:00:00,1,1


In [8]:
demand["hour"] = demand["pickup_hour"].dt.hour
demand["day_of_week"] = demand["pickup_hour"].dt.dayofweek
demand["month"] = demand["pickup_hour"].dt.month

In [9]:
demand.head()

,pickup_hour,PULocationID,pickup_count,hour,day_of_week,month
2,2022-10-24 17:00:00,1,1,17,0,10
8,2022-10-25 03:00:00,1,1,3,1,10
623,2023-01-01 05:00:00,1,1,5,6,1
916,2023-01-01 08:00:00,1,1,8,6,1
1224,2023-01-01 11:00:00,1,1,11,6,1


In [10]:
demand["is_weekend"] = demand["day_of_week"].isin([5, 6]).astype(int)

In [11]:
demand["lag_1"] = demand.groupby("PULocationID")["pickup_count"].shift(1)

demand["lag_24"] = demand.groupby("PULocationID")["pickup_count"].shift(24)

In [12]:
demand["rolling_24"] = (
    demand.groupby("PULocationID")["pickup_count"]
    .shift(1)
    .rolling(24)
    .mean()
)

In [13]:
demand = demand.dropna()

In [14]:
print(demand.head())

             pickup_hour  PULocationID  pickup_count  hour  day_of_week  \
3667 2023-01-02 14:00:00             1             4    14            0   
3775 2023-01-02 15:00:00             1             3    15            0   
3987 2023-01-02 17:00:00             1             2    17            0   
4089 2023-01-02 18:00:00             1             1    18            0   
4178 2023-01-02 19:00:00             1             1    19            0   

      month  is_weekend  lag_1  lag_24  rolling_24  
3667      1           0    6.0     1.0    2.500000  
3775      1           0    4.0     1.0    2.625000  
3987      1           0    3.0     1.0    2.708333  
4089      1           0    2.0     1.0    2.750000  
4178      1           0    1.0     1.0    2.750000  


In [15]:
features = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_1",
    "lag_24",
    "rolling_24"
]

target = "pickup_count"

In [16]:
split_date = demand["pickup_hour"].quantile(0.8)

train = demand[demand["pickup_hour"] <= split_date]
test = demand[demand["pickup_hour"] > split_date]

In [17]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=50, random_state=42)

model.fit(train[features], train[target])

RandomForestRegressor(n_estimators=50, random_state=42)

In [18]:
from sklearn.metrics import mean_squared_error
import numpy as np

preds = model.predict(test[features])

rmse = np.sqrt(mean_squared_error(test[target], preds))

print("RMSE:", rmse)

RMSE: 15.897426890987287


In [19]:
import joblib

joblib.dump(model, "../models/model.joblib")

['../models/model.joblib']